<br>

# Transform


In [ ]:
import pandas as pd

from tjsp_unidades import extract
from tjsp_unidades.transform.small_functions import adjust_columns

<br>

---

## Get Data


<br>

---

### Quem Somos


In [ ]:
quem_somos = extract.QuemSomos()

quem_somos.rajs.head()

In [ ]:
quem_somos.cjs.head()

In [ ]:
quem_somos.comarcas

<br>

---

### Municípios


In [ ]:
listar_municipios = extract.ListarMunicipios()

# Obtem todos os IDs dos Municípios do TJSP
df_municipios = listar_municipios.request()

In [ ]:
# Results
df_municipios.info()
df_municipios.head()

<br>

---

### Unidades


In [ ]:
listar_unidades = extract.ListarUnidades()

df_unidades = listar_unidades.get_unidades_batch(
    list_id_tjsp=list(df_municipios["id_municipio_tjsp"]),
)

df_unidades.head()

<br>

---

## Tratamento


In [ ]:
# Merge
df_unidades_merge = pd.merge(
    left=df_municipios,
    right=df_unidades,
    left_on="id_municipio_tjsp",
    right_on="id_municipio_tjsp",
    how="inner",
    suffixes=["", "_copy"],
)

df_unidades_merge = df_unidades_merge.drop(
    labels=["municipio_tjsp_copy"],
    axis="columns",
    errors="ignore",
)

# Unidades
df_unidades_merge.info()
df_unidades_merge.head()

<br>

---

### Comarcas


In [ ]:
df_tjsp = df_unidades_merge.copy()

# Comarca
df_tjsp_com = df_tjsp[df_tjsp["comarca_sede"] == 1]

# Filtra Colunas
df_tjsp_com = df_tjsp_com.drop(
    labels=[
        "comarca_sede",
        "unidades",
        "raj",
    ],
    axis="columns",
    errors="ignore",
)

# Deleta Duplicados
df_tjsp_com = df_tjsp_com.drop_duplicates()

# Renomeia Colunas
df_tjsp_com = df_tjsp_com.rename(
    {
        "municipio_tjsp_corrigido": "comarca_tjsp_corrigido",
        "id_municipio_ibge": "id_comarca",
    },
    axis="columns",
)

# Deleta
df_tjsp_com = df_tjsp_com.drop(
    labels=["municipio_tjsp"],
    axis="columns",
    errors="ignore",
)

# Bata Bater
df_tjsp_com = adjust_columns(
    df=df_tjsp_com,
    column_ajust="comarca_tjsp",
)

df_tjsp_com.info()
df_tjsp_com.head()

In [ ]:
df_temp = adjust_columns(df=quem_somos.comarcas, column_ajust="comarca_tjsp")

Usando `left` eu dropo a Vila Mimosa!


In [ ]:
df_tjsp_com = pd.merge(
    left=df_tjsp_com,
    right=df_temp,
    left_on="comarca_tjsp_temp",
    right_on="comarca_tjsp_temp",
    how="left",
    suffixes=["", "_copy"],
)

# Deleta
df_tjsp_com = df_tjsp_com.drop(
    labels=["comarca_tjsp_copy"],
    axis="columns",
    errors="ignore",
)

# Ordena
df_tjsp_com = df_tjsp_com.iloc[
    df_tjsp_com["comarca_tjsp"].str.normalize("NFKD").argsort()
]

# Reset Index
df_tjsp_com = df_tjsp_com.reset_index(drop=True)

df_tjsp_com = df_tjsp_com.drop(
    labels=["comarca_tjsp_temp", "id_municipio_tjsp"],
    axis="columns",
)

# Reordena
df_tjsp_com = df_tjsp_com[
    [
        "id_comarca",
        "comarca_tjsp",
        "comarca_tjsp_corrigido",
        "id_cj",
    ]
]

df_temp = df_tjsp_com[df_tjsp_com["id_cj"].isna()]
if len(df_temp) > 0:
    raise Exception("Erro")

df_tjsp_com.info()
df_tjsp_com.head()

<br>

---

### Municipios > Comarcas


In [ ]:
# Filtra Colunas
df_municipios_fim = df_unidades_merge.drop(
    labels=[
        "unidades",
        "raj",
        #'id_municipio_tjsp'
    ],
    axis="columns",
    errors="ignore",
)

# Deleta Duplicados
df_municipios_fim = df_municipios_fim.drop_duplicates()

# Reset Index
df_municipios_fim = df_municipios_fim.reset_index(drop=True)

df_municipios_fim.info()
df_municipios_fim.head()

In [ ]:
df_temp = adjust_columns(df=df_municipios_fim, column_ajust="comarca_tjsp")
df_temp.head()

In [ ]:
df_temp2 = adjust_columns(df=df_tjsp_com, column_ajust="comarca_tjsp")

# Filtra Colunas
df_temp2 = df_temp2.drop(
    labels=[
        "comarca_tjsp",
        "comarca_tjsp_corrigido",
        "id_cj",
    ],
    axis="columns",
    errors="ignore",
)

df_temp2.head()

In [ ]:
# Merge
df_temp3 = pd.merge(
    left=df_temp,
    right=df_temp2,
    left_on="comarca_tjsp_temp",
    right_on="comarca_tjsp_temp",
    how="left",
    suffixes=["", "_copy"],
)

# Filtra Colunas
df_temp3 = df_temp3.drop(
    labels=[
        "comarca_tjsp_temp",
        "comarca_tjsp",
    ],
    axis="columns",
    errors="ignore",
)

df_temp = df_temp3[df_temp3["id_comarca"].isna()]
if len(df_temp) > 0:
    raise Exception("Erro")

# Results
df_temp3.info()
df_temp3.head()

<br>

---

### Unidades


In [ ]:
# Unidades
df_unidades_merge.info()
df_unidades_merge.head()

In [ ]:
df_tjsp_unidades = df_unidades_merge[["id_municipio_ibge", "imoveis"]]
df_tjsp_unidades = df_tjsp_unidades.drop_duplicates()
df_tjsp_unidades